In [0]:
print("===================================")
print("CUSTOMER SEGMENTATION PROJECT")
print("===================================")

print("Platform: Databricks")
print("Language: Python")
print("Processing: PySpark")
print("Storage: Delta Lake")

print("Project setup successful!")

CUSTOMER SEGMENTATION PROJECT
Platform: Databricks
Language: Python
Processing: PySpark
Storage: Delta Lake
Project setup successful!


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
# Project configuration

SEED = 20261022

CUSTOMER_COUNT = 1000

print("Random Seed:", SEED)
print("Number of Customers:", CUSTOMER_COUNT)

Random Seed: 20261022
Number of Customers: 1000


In [0]:
# Generate customer IDs

customers = spark.range(1, CUSTOMER_COUNT + 1)

customers = customers.withColumn(
    "customer_id",
    F.concat(
        F.lit("CUS-"),
        F.lpad(F.col("id").cast("string"), 6, "0")
    )
)



In [0]:
# Generate deterministic values
# Every attribute gets its own hash, so the attributes are independent of each other.
# (Deriving all of them from one number with % made gender fully determined by channel.)

def h(name, modulus):
    return F.pmod(F.xxhash64("customer_id", F.lit(f"{SEED}-{name}")), F.lit(modulus))

customers = customers.withColumn(
    "age",
    (h("age", 48) + 18).cast("int")
)

customers = customers.withColumn(
    "gender",
    F.when(h("gender", 2) == 0, "Female")
     .otherwise("Male")
)

customers = customers.withColumn(
    "city",
    F.when(h("city", 5) == 0, "Bangalore")
     .when(h("city", 5) == 1, "Mumbai")
     .when(h("city", 5) == 2, "Delhi")
     .when(h("city", 5) == 3, "Hyderabad")
     .otherwise("Chennai")
)

customers = customers.withColumn(
    "acquisition_channel",
    F.when(h("channel", 4) == 0, "Website")
     .when(h("channel", 4) == 1, "Mobile App")
     .when(h("channel", 4) == 2, "Referral")
     .otherwise("Social Media")
)

customers = customers.withColumn(
    "signup_date",
    F.date_add(
        F.lit("2024-01-01").cast("date"),
        h("signup", 1033).cast("int")
    )
)

customers = customers.withColumn(
    "account_status",
    F.lit("ACTIVE")
)


In [0]:
# Select final columns

customer_df = customers.select(
    "customer_id",
    "age",
    "gender",
    "city",
    "signup_date",
    "acquisition_channel",
    "account_status"
)

print("Customer data generated successfully!")


Customer data generated successfully!


In [0]:
#Display the customer data
customer_df.show(10, truncate=False)

+-----------+---+------+---------+-----------+-------------------+--------------+
|customer_id|age|gender|city     |signup_date|acquisition_channel|account_status|
+-----------+---+------+---------+-----------+-------------------+--------------+
|CUS-000001 |64 |Female|Mumbai   |2024-02-29 |Referral           |ACTIVE        |
|CUS-000002 |54 |Female|Mumbai   |2024-02-22 |Social Media       |ACTIVE        |
|CUS-000003 |18 |Female|Chennai  |2024-11-09 |Social Media       |ACTIVE        |
|CUS-000004 |26 |Female|Chennai  |2024-11-12 |Referral           |ACTIVE        |
|CUS-000005 |64 |Male  |Delhi    |2026-07-16 |Referral           |ACTIVE        |
|CUS-000006 |34 |Male  |Mumbai   |2025-11-13 |Referral           |ACTIVE        |
|CUS-000007 |47 |Female|Mumbai   |2026-07-04 |Referral           |ACTIVE        |
|CUS-000008 |18 |Female|Mumbai   |2024-06-02 |Social Media       |ACTIVE        |
|CUS-000009 |55 |Male  |Bangalore|2024-07-18 |Website            |ACTIVE        |
|CUS-000010 |49 

In [0]:
#Check the customer count
print("Total Customers:", customer_df.count())

customer_df.printSchema()

Total Customers: 1000
root
 |-- customer_id: string (nullable = false)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = false)
 |-- city: string (nullable = false)
 |-- signup_date: date (nullable = true)
 |-- acquisition_channel: string (nullable = false)
 |-- account_status: string (nullable = false)



In [0]:
#Validate customer IDs
# Check duplicate customer IDs

duplicate_count = (
    customer_df.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate Customer IDs:", duplicate_count)

Duplicate Customer IDs: 0


In [0]:
#Save customer data in Delta Lake
customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_master")

In [0]:
%sql
SELECT * FROM customer_master LIMIT 10;

customer_id,age,gender,city,signup_date,acquisition_channel,account_status
CUS-000751,20,Male,Mumbai,2026-06-11,Website,ACTIVE
CUS-000752,18,Female,Bangalore,2026-10-14,Website,ACTIVE
CUS-000753,32,Female,Delhi,2025-11-08,Social Media,ACTIVE
CUS-000754,19,Male,Bangalore,2026-08-11,Website,ACTIVE
CUS-000755,35,Female,Hyderabad,2024-10-08,Mobile App,ACTIVE
CUS-000756,41,Male,Bangalore,2024-03-10,Website,ACTIVE
CUS-000757,37,Male,Hyderabad,2025-11-13,Mobile App,ACTIVE
CUS-000758,62,Female,Mumbai,2024-05-19,Social Media,ACTIVE
CUS-000759,60,Female,Bangalore,2025-08-10,Mobile App,ACTIVE
CUS-000760,18,Female,Delhi,2026-03-16,Website,ACTIVE
